# Mesa de Partes Digital-Sismo (Caso Ate) — Versión Evaluación Final (EF)

In [ ]:
import os
os.makedirs("/content/Mesa_Partes_Sismo_Demo", exist_ok=True)
%cd /content/Mesa_Partes_Sismo_Demo


## 1. Validaciones (`validaciones.py`)

In [ ]:
%%writefile validaciones.py
"""Módulo de validaciones para las solicitudes ciudadanas de Ate."""
from datetime import datetime

# Distritos habilitados en el prototipo (alcance acotado al distrito de Ate)
DISTRITOS = ["ATE"]


def validar_dni(dni):
    """Comprueba que el DNI tenga exactamente 8 dígitos numéricos."""
    return len(dni) == 8 and dni.isdigit()


def validar_nombre(nombre):
    """Comprueba que el nombre tenga al menos 3 caracteres, solo letras y espacios."""
    limpio = nombre.strip()
    if len(limpio) < 3:
        return False
    for letra in limpio:
        if not (letra.isalpha() or letra == " "):
            return False
    return True


def validar_distrito(distrito):
    """Comprueba que el distrito ingresado corresponda al alcance del prototipo (ATE)."""
    return distrito.strip().upper() in DISTRITOS


def validar_tipo(tipo):
    """Comprueba que el tipo de solicitud sea 1 (Daño), 2 (Ayuda) o 3 (Atención)."""
    return tipo in ("1", "2", "3")


def validar_descripcion(texto):
    """Comprueba longitud entre 10 y 150 caracteres y bloquea '|' para proteger el TXT."""
    return 10 <= len(texto.strip()) <= 150 and "|" not in texto


def validar_personas(texto):
    """Comprueba que la cantidad de personas afectadas sea un entero entre 0 y 500."""
    return texto.isdigit() and 0 <= int(texto) <= 500


def validar_si_no(texto):
    """Comprueba que la respuesta sea 'S' o 'N' sin importar mayúsculas o espacios."""
    return texto.strip().upper() in ("S", "N")


def validar_fecha(fecha):
    """Comprueba que la fecha exista en el calendario con formato dd/mm/aaaa."""
    try:
        datetime.strptime(fecha, "%d/%m/%Y")
        return True
    except ValueError:
        return False


## 2. Clasificación (`clasificacion.py`)

In [ ]:
%%writefile clasificacion.py
"""Módulo de clasificación de expedientes en cuatro niveles de prioridad."""

# Palabras clave que indican riesgo inmediato en la descripción del reporte
PALABRAS_GRAVES = [
    "atrapado", "atrapada", "derrumbe", "colapso", "incendio",
    "fuga de gas", "herido", "sin respirar", "desaparecido",
]

# Rango numérico para ordenar de mayor gravedad (1) a menor gravedad (4)
RANGO = {"CRITICO": 1, "ALTO": 2, "MEDIO": 3, "BAJO": 4}


def contiene_palabra_grave(descripcion):
    """Retorna True si la descripción incluye alguna expresión de gravedad."""
    texto = descripcion.lower()
    for palabra in PALABRAS_GRAVES:
        if palabra in texto:
            return True
    return False


def calcular_puntaje(tipo, personas, heridos, descripcion):
    """Calcula el puntaje discreto de gravedad (entre 1 y 12 puntos)."""
    puntaje = 0

    # Puntos por tipo de solicitud (Ayuda=3, Daño=2, Atención=1)
    if tipo == "2":
        puntaje += 3
    elif tipo == "1":
        puntaje += 2
    else:
        puntaje += 1

    # Puntos por cantidad de personas afectadas
    if personas >= 10:
        puntaje += 3
    elif personas >= 4:
        puntaje += 2
    elif personas >= 1:
        puntaje += 1

    # Puntos adicionales por heridos confirmados y palabras de gravedad
    if heridos == "S":
        puntaje += 3
    if contiene_palabra_grave(descripcion):
        puntaje += 3

    return puntaje


def clasificar(puntaje):
    """Asigna el nivel de prioridad según intervalos disjuntos del puntaje total."""
    if puntaje >= 8:
        return "CRITICO"
    if puntaje >= 5:
        return "ALTO"
    if puntaje >= 3:
        return "MEDIO"
    return "BAJO"


## 3. Expedientes (`expedientes.py`)

In [ ]:
%%writefile expedientes.py
"""Módulo de operaciones sobre expedientes: registro, búsqueda, ordenamiento y resumen."""
import clasificacion as cla

# Mapa de códigos numéricos del menú a etiquetas legibles de tipo de solicitud
TIPOS = {"1": "Daño", "2": "Ayuda", "3": "Atención"}


def generar_codigo(codigos, anio):
    """Genera un código correlativo único con formato SIS-AAAA-NNNN."""
    return f"SIS-{anio}-{len(codigos) + 1:04d}"


def registrar(datos, dni, nombre, distrito, tipo, descripcion, personas, heridos, fecha, anio):
    """Clasifica un nuevo reporte y lo inserta sincronizado en las 10 listas paralelas."""
    codigo = generar_codigo(datos["codigos"], anio)
    puntaje = cla.calcular_puntaje(tipo, int(personas), heridos.upper(), descripcion)
    prioridad = cla.clasificar(puntaje)

    # Inserción sincronizada: el mismo índice i representa un único expediente
    datos["codigos"].append(codigo)
    datos["dnis"].append(dni)
    datos["nombres"].append(nombre.strip().title())
    datos["distritos"].append(distrito.strip().upper())
    datos["tipos"].append(TIPOS[tipo])
    datos["descripciones"].append(descripcion.strip())
    datos["personas"].append(str(int(personas)))
    datos["heridos"].append(heridos.upper())
    datos["fechas"].append(fecha)
    datos["prioridades"].append(prioridad)
    return codigo, prioridad


def buscar_por_codigo(codigos, codigo):
    """Realiza una búsqueda secuencial por código y retorna su índice o -1 si no existe."""
    objetivo = codigo.strip().upper()
    for i in range(len(codigos)):
        if codigos[i] == objetivo:
            return i
    return -1


def _burbuja(claves, datos):
    """Ordena usando método burbuja intercambiando en sincronía las 10 listas paralelas."""
    n = len(claves)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if claves[j] > claves[j + 1]:
                claves[j], claves[j + 1] = claves[j + 1], claves[j]
                for campo in datos:
                    arreglo = datos[campo]
                    arreglo[j], arreglo[j + 1] = arreglo[j + 1], arreglo[j]


def ordenar_por_codigo(datos):
    """Ordena los expedientes ascendentemente por su código identificador."""
    _burbuja(list(datos["codigos"]), datos)


def ordenar_por_prioridad(datos):
    """Ordena por doble criterio: primero por código y luego por nivel de prioridad."""
    ordenar_por_codigo(datos)
    claves = []
    for prioridad in datos["prioridades"]:
        claves.append(cla.RANGO[prioridad])
    _burbuja(claves, datos)


def contar_por_prioridad(prioridades):
    """Cuenta la cantidad de expedientes registrados por cada nivel de prioridad."""
    conteo = {"CRITICO": 0, "ALTO": 0, "MEDIO": 0, "BAJO": 0}
    for p in prioridades:
        if p in conteo:
            conteo[p] += 1
    return conteo


def contar_por_tipo(tipos):
    """Cuenta la cantidad de expedientes registrados por cada tipo de solicitud."""
    conteo = {"Ayuda": 0, "Daño": 0, "Atención": 0}
    for t in tipos:
        if t in conteo:
            conteo[t] += 1
    return conteo


def sumar_afectados_recursivo(personas, indice=0):
    """Suma de forma recursiva el total de personas afectadas en el arreglo paralelo."""
    if indice == len(personas):
        return 0
    return int(personas[indice]) + sumar_afectados_recursivo(personas, indice + 1)


## 4. Archivos (`archivos.py`)

In [ ]:
%%writefile archivos.py
"""Módulo de persistencia para guardar y recuperar expedientes en archivo TXT."""
import os

# Ruta del archivo de texto plano y orden fijo de los 10 campos paralelos
RUTA = "reportes_sismo.txt"
CAMPOS = [
    "codigos", "dnis", "nombres", "distritos", "tipos",
    "descripciones", "personas", "heridos", "fechas", "prioridades",
]


def nuevo_almacen():
    """Inicializa el contenedor con 10 listas paralelas vacías (una por campo)."""
    datos = {}
    for campo in CAMPOS:
        datos[campo] = []
    return datos


def guardar(datos, ruta=RUTA):
    """Escribe todos los expedientes en formato texto plano delimitado por '|' (UTF-8)."""
    with open(ruta, "w", encoding="utf-8") as archivo:
        for i in range(len(datos["codigos"])):
            campos_reporte = []
            for campo in CAMPOS:
                campos_reporte.append(datos[campo][i])
            archivo.write("|".join(campos_reporte) + "\n")


def cargar(ruta=RUTA):
    """Recupera los expedientes desde el archivo TXT; retorna listas vacías si no existe."""
    datos = nuevo_almacen()
    if not os.path.exists(ruta):
        return datos
    with open(ruta, "r", encoding="utf-8") as archivo:
        for linea in archivo:
            partes = linea.rstrip("\n").split("|")
            if len(partes) == len(CAMPOS):
                for i in range(len(CAMPOS)):
                    datos[CAMPOS[i]].append(partes[i])
    return datos


## 5. Menú principal y gráficos (`main.py`)

In [ ]:
%%writefile main.py
"""Módulo principal: menú interactivo de la Mesa de Partes Digital-Sismo (Ate)."""
from datetime import datetime
import validaciones as val
import expedientes as exp
import archivos as arc


def pedir_dato(mensaje, tipo, error):
    """Solicita un dato por consola y repite hasta cumplir su regla de validación."""
    while True:
        dato = input(mensaje).strip()
        correcto = False

        if tipo == "dni":
            correcto = val.validar_dni(dato)
        elif tipo == "nombre":
            correcto = val.validar_nombre(dato)
        elif tipo == "distrito":
            correcto = val.validar_distrito(dato)
        elif tipo == "reporte":
            correcto = val.validar_tipo(dato)
        elif tipo == "descripcion":
            correcto = val.validar_descripcion(dato)
        elif tipo == "personas":
            correcto = val.validar_personas(dato)
        elif tipo == "heridos":
            correcto = val.validar_si_no(dato)
        elif tipo == "fecha":
            # Mejora de usabilidad: Enter toma automáticamente la fecha actual
            if dato == "":
                return datetime.now().strftime("%d/%m/%Y")
            correcto = val.validar_fecha(dato)

        if correcto:
            return dato
        print(" [!] " + error)


def mostrar(datos, i):
    """Muestra en una línea legible los datos principales del expediente en la posición i."""
    print(
        f" [{datos['prioridades'][i]}] {datos['codigos'][i]} | "
        f"{datos['nombres'][i]} | {datos['distritos'][i]} | "
        f"{datos['tipos'][i]} | {datos['personas'][i]} personas | "
        f"{datos['descripciones'][i]}"
    )


def mostrar_grafico_resumen(conteo_prioridad, conteo_tipo):
    """Genera un panel doble de barras (prioridad y tipo de solicitud) usando matplotlib."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print(" [Aviso] matplotlib no está instalado; se muestra solo el resumen en texto.")
        return

    niveles = list(conteo_prioridad.keys())
    valores_prio = list(conteo_prioridad.values())
    colores_prio = ["#D32F2F", "#F57C00", "#FBC02D", "#388E3C"]

    tipos = list(conteo_tipo.keys())
    valores_tipo = list(conteo_tipo.values())
    colores_tipo = ["#1565C0", "#6A1B9A", "#00838F"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # Gráfico 1: Expedientes por nivel de prioridad
    barras1 = ax1.bar(niveles, valores_prio, color=colores_prio)
    ax1.set_title("Expedientes por Nivel de Prioridad")
    ax1.set_xlabel("Prioridad")
    ax1.set_ylabel("Cantidad de expedientes")
    ax1.set_ylim(0, max(valores_prio + [1]) + 1)
    for barra in barras1:
        alto = int(barra.get_height())
        ax1.text(barra.get_x() + barra.get_width() / 2, alto + 0.1, str(alto), ha="center")

    # Gráfico 2: Expedientes por tipo de solicitud
    barras2 = ax2.bar(tipos, valores_tipo, color=colores_tipo)
    ax2.set_title("Expedientes por Tipo de Solicitud")
    ax2.set_xlabel("Tipo")
    ax2.set_ylabel("Cantidad de expedientes")
    ax2.set_ylim(0, max(valores_tipo + [1]) + 1)
    for barra in barras2:
        alto = int(barra.get_height())
        ax2.text(barra.get_x() + barra.get_width() / 2, alto + 0.1, str(alto), ha="center")

    fig.suptitle("Resumen Operativo — Mesa de Partes Digital-Sismo (Ate)", fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def menu():
    """Bucle principal del sistema: carga los expedientes y gestiona las 6 opciones."""
    datos = arc.cargar()
    anio = datetime.now().year
    opcion = ""

    while opcion != "6":
        print("\n=== MESA DE PARTES DIGITAL-SISMO (ATE) ===")
        print("1. Registrar     2. Buscar por código     3. Ordenar por prioridad")
        print("4. Listar        5. Resumen               6. Salir")
        opcion = input("Opción: ").strip()

        if opcion == "1":
            hoy = datetime.now().strftime("%d/%m/%Y")
            dni = pedir_dato("DNI (8 dígitos): ", "dni", "DNI inválido.")
            nombre = pedir_dato("Nombre completo: ", "nombre", "Nombre inválido.")
            distrito = pedir_dato("Distrito: ", "distrito", "Por ahora solo se registra ATE.")
            tipo = pedir_dato("Tipo (1 Daño, 2 Ayuda, 3 Atención): ", "reporte", "Elija 1, 2 o 3.")
            desc = pedir_dato("Descripción (10-150 caracteres): ", "descripcion", "Descripción inválida.")
            personas = pedir_dato("Personas afectadas: ", "personas", "Número de 0 a 500.")
            heridos = pedir_dato("¿Hay heridos? (S/N): ", "heridos", "Responda S o N.")
            fecha = pedir_dato(
                f"Fecha (dd/mm/aaaa, Enter para hoy {hoy}): ",
                "fecha",
                "Fecha inválida.",
            )
            codigo, prioridad = exp.registrar(
                datos, dni, nombre, distrito, tipo, desc, personas, heridos, fecha, anio
            )
            arc.guardar(datos)
            print(f" Registrado: {codigo} | Prioridad: {prioridad} | Fecha: {fecha}")

        elif opcion == "2":
            i = exp.buscar_por_codigo(datos["codigos"], input("Código: "))
            if i == -1:
                print(" Expediente no encontrado.")
            else:
                mostrar(datos, i)

        elif opcion == "3":
            exp.ordenar_por_prioridad(datos)
            arc.guardar(datos)
            print(" Ordenado: casos críticos primero.")

        elif opcion == "4":
            if not datos["codigos"]:
                print(" No hay expedientes.")
            for i in range(len(datos["codigos"])):
                mostrar(datos, i)

        elif opcion == "5":
            conteo = exp.contar_por_prioridad(datos["prioridades"])
            conteo_tipos = exp.contar_por_tipo(datos["tipos"])
            total_afectados = exp.sumar_afectados_recursivo(datos["personas"])

            print("\n--- Resumen por Prioridad ---")
            for nivel in conteo:
                print(f" {nivel}: {conteo[nivel]}")
            print("\n--- Resumen por Tipo de Solicitud ---")
            for tipo_nombre in conteo_tipos:
                print(f" {tipo_nombre}: {conteo_tipos[tipo_nombre]}")
            print(f"\n Total de personas afectadas: {total_afectados}")

            if datos["codigos"]:
                mostrar_grafico_resumen(conteo, conteo_tipos)
            else:
                print(" No hay expedientes suficientes para generar gráficos.")

        elif opcion != "6":
            print(" Opción no válida.")


if __name__ == "__main__":
    menu()


## 6. Diez expedientes de demostración (`casos_demo_sismo_10.txt`)

In [ ]:
%%writefile casos_demo_sismo_10.txt
SIS-2026-0001|00000001|Ana Prueba|ATE|Atención|Consulta sobre el punto de informacion vecinal|0|N|21/10/2026|BAJO
SIS-2026-0002|00000002|Luis Ejemplo|ATE|Daño|Grietas en una pared de la vivienda|1|N|21/10/2026|MEDIO
SIS-2026-0003|00000003|Maria Simulada|ATE|Ayuda|Familias atrapadas en edificio con colapso|12|S|21/10/2026|CRITICO
SIS-2026-0004|00000004|Carlos Ensayo|ATE|Ayuda|Se requieren agua y mantas para cinco vecinos|5|N|21/10/2026|ALTO
SIS-2026-0005|00000005|Rosa Ficticia|ATE|Atención|Pregunta por horario de atencion municipal|0|N|21/10/2026|BAJO
SIS-2026-0006|00000006|Pedro Prueba|ATE|Daño|Derrumbe parcial cerca de viviendas afectadas|9|S|21/10/2026|CRITICO
SIS-2026-0007|00000007|Elena Ejemplo|ATE|Atención|Consultan por la ubicacion del punto de apoyo|4|N|21/10/2026|MEDIO
SIS-2026-0008|00000008|Jorge Simulado|ATE|Daño|Viviendas con grietas y otros daños materiales|11|N|21/10/2026|ALTO
SIS-2026-0009|00000009|Lucia Ensayo|ATE|Ayuda|Solicitan informacion sobre reparto de alimentos|0|N|21/10/2026|MEDIO
SIS-2026-0010|00000010|Diana Ficticia|ATE|Ayuda|Vecino herido necesita traslado inmediato|2|S|21/10/2026|CRITICO


## 7. Cargar la demostración una sola vez

In [ ]:
import os
import shutil

if not os.path.exists("reportes_sismo.txt"):
    shutil.copyfile("casos_demo_sismo_10.txt", "reportes_sismo.txt")
    print("Demostración preparada con 10 expedientes ficticios.")
else:
    print("Se conserva el archivo de trabajo de esta sesión.")

with open("reportes_sismo.txt", "r", encoding="utf-8") as archivo:
    cantidad = len(archivo.readlines())
print("Expedientes disponibles:", cantidad)


## 8. Ejecutar el prototipo

In [ ]:
%run main.py


## 9. Descargar respaldo de expedientes (`reportes_sismo.txt`)

In [ ]:
from google.colab import files
files.download("reportes_sismo.txt")


## 10. Restauración manual a los 10 casos iniciales (solo si desea reiniciar la demo)
```python
import shutil
shutil.copyfile("casos_demo_sismo_10.txt", "reportes_sismo.txt")
```